In [ ]:
import os
os.getcwd()

In [ ]:
Share_point = r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2021-11 European natural gas imports\Code\Imports'

In [ ]:
os.chdir(Share_point)

In [ ]:
import json
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.dates import DateFormatter
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

import eurostat  # python wrapper for taking data.
import time

In [ ]:
# Converter for metric tonnes to M3m 
# converter = -0.001397 WRONG
converter = -0.001379

### In Million tonnes

In [ ]:
d1 = pd.read_excel('raw_data/LNG/Europe LNG 2017_2018.xlsx',index_col=1)
d2 = pd.read_excel('raw_data/LNG/Europe LNG 2019_2020.xlsx',index_col=1)
d3 = pd.read_excel('raw_data/LNG/Europe LNG 2021_2022.xlsx',index_col=1)
d4 = pd.read_excel('raw_data/LNG/Europe LNG 2023.xlsx',index_col=1)

In [ ]:
# Merge 2017 to 2020
df_20 = pd.merge(d1,d2,on=['Origin_Berthcountry','Port_Country'],how='outer')

In [ ]:
# Merge 2017 to 2022
df_22 = pd.merge(df_20,d3,on=['Origin_Berthcountry','Port_Country'],how='outer')

In [ ]:
# Merge 2022 to 2023
df = pd.merge(df_22,d4,on=['Origin_Berthcountry','Port_Country'],how='outer')

In [ ]:
not_EU_ES_PT = ['United Kingdom','Turkey','Spain','Portugal']
df_EU25 = df.loc[~df['Port_Country'].isin((not_EU_ES_PT))] 

### Single-out Spain and Portugal

In [ ]:
iberia = df[df['Port_Country'].isin(['Spain','Portugal'])].reset_index()

In [ ]:
cols = iberia.iloc[:,:2]

In [ ]:
# convert to M3m
values = iberia.iloc[:,2:]*converter

In [ ]:
iberia = pd.concat([cols,values], axis=1)

In [ ]:
iberia = iberia.groupby(['Origin_Berthcountry']).sum(numeric_only=True)

In [ ]:
iberia = iberia.T #drop(

In [ ]:
iberia.index = pd.to_datetime(iberia.index)
iberia = iberia.sort_index()

In [ ]:
iberia.head()

In [ ]:
include_i = ['Algeria','Norway','Nigeria','Qatar','Russia','Trinidad & Tobago','United States of America','Egypt']
exclude_i = list(set(iberia.columns)-set(include_i))

In [ ]:
## Define an "Other" category
iberia['Other']= iberia[exclude_i].sum(axis=1)

In [ ]:
iberia_f = iberia[['Algeria','Norway','Nigeria','Qatar','Russia','Trinidad & Tobago','United States of America','Egypt','Other']]

In [ ]:
iberia_f=iberia_f.rename(columns= {'United States of America':'United States'})

In [ ]:
iberia_f = iberia_f[['United States','Russia', 'Qatar',  'Norway', 'Algeria', 'Nigeria', 'Trinidad & Tobago', 'Egypt', 'Other']]

In [ ]:
iberia_twh = iberia_f*10.3/1000

In [ ]:
iberia_twh.plot(figsize=(15, 9),kind='area',ylabel='TWh', title='LNG imports in the Iberian peninsula by supplier')

In [ ]:
iberia_twh.to_excel(r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2023-04 How the EU can phase out Russian LNG\Data\Iberian LNG by source in TWh.xlsx')

## Back to recurrent code

In [ ]:
## Here we get 
dff = df_EU25.groupby(['Origin_Berthcountry']).sum(numeric_only=True)

In [ ]:
dff_T = dff.T #drop(['nan_x','nan_y'])

In [ ]:
dff_T.index = pd.to_datetime(dff_T.index)
dff_T = dff_T.sort_index()

### in M3m

In [ ]:
dff_M3m = dff_T * converter

In [ ]:
# Remove European re-exports
del dff_M3m['Belgium']
del dff_M3m['France']
del dff_M3m['Lithuania']
del dff_M3m['Netherlands']
del dff_M3m['Spain']

In [ ]:
dff_EU = dff_M3m

In [ ]:
dff_M3m.columns

In [ ]:
dff_M3m.tail()

In [ ]:
include = ['Algeria','Norway','Nigeria','Qatar','Russia','Trinidad & Tobago','United States of America','Egypt']
exclude = list(set(dff_M3m.columns)-set(include))

In [ ]:
## Define an "Other" category
dff_M3m['Other']= dff_M3m[exclude].sum(axis=1)

In [ ]:
# dff_M3m['Argentina'].loc['2020':]+dff_M3m['Brazil'].loc['2020':]+dff_M3m['Peru'].loc['2020':]

In [ ]:
dff_M3m['month'] = dff_M3m.index.strftime("%m")

In [ ]:
os.getcwd()

In [ ]:
dff_M3m

In [ ]:
today = date.today()    

#### Get only 2019 onwards values

In [ ]:
df_late =dff_M3m.loc['2019':]

In [ ]:
df_late=df_late.rename(columns={'United States of America':'United States'})

In [ ]:
df_late=df_late.drop('month',axis=1)
df_late = df_late[['Qatar','Algeria','Nigeria','Russia','Norway','United States','Trinidad & Tobago','Egypt','Other']]

In [ ]:
import os
os.getcwd()

In [ ]:
# df_late.to_excel(r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2023-04 How the EU can phase out Russian LNG\Data\granular EU25 LNG imports.xlsx')

#### Aggregate to avoid problems with Bloomberg

In [ ]:
dff_EU = dff_EU.loc['2019':]

In [ ]:
dff_EU.columns

### In TWh

In [ ]:
df_late.columns

In [ ]:
df_twh = df_late/1000*10.3

In [ ]:
df_twh = df_twh[['United States','Russia', 'Qatar',  'Norway', 'Algeria', 'Nigeria', 'Trinidad & Tobago', 'Egypt', 'Other']]

In [ ]:
df_twh.to_excel(r'C:\Users\giovanni.sgaravatti\Bruegel\Research - 2023-04 How the EU can phase out Russian LNG\Data\EU25 LNG imports.xlsx')

In [ ]:
colors = {'United States':'#0080C7','Russia':'#E67425', 'Qatar':'#FFC000',  'Norway':'black', 'Algeria':'orange', 'Nigeria':'#2BB0CB', 'Trinidad & Tobago':'#0080C7', 'Egypt':'blue', 'Other':'#58A944'}

In [ ]:
df_twh.plot(figsize=(15, 9),kind='area',ylabel='TWh', title='LNG imports in the EU by supplier',color=colors)